# Commodity Regime Shifts: Adaptive Bandits on Real-World Patterns

**Goal:** Apply non-stationary bandits to commodity-like data with historical regime shifts.

**Real-world events we'll simulate:**
- COVID-19 crash (March 2020): Oil collapse
- 2022 Energy crisis: Natural gas spike, supply chain chaos
- Flight to safety patterns: Gold rallies during uncertainty

**Key insight:** In commodities, non-stationarity isn't the exception — it's the rule.

## Setup: Load Commodity-Like Data

We'll create synthetic data that mimics real commodity behavior with regime changes.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import beta
from datetime import datetime, timedelta

np.random.seed(42)

def generate_commodity_returns(start_date='2019-01-01', days=1000):
    """
    Generate synthetic commodity returns with regime structure.

    Regimes:
    1. Normal (days 0-250): Balanced performance
    2. COVID crash (days 250-400): Oil collapses, gold rallies
    3. Recovery (days 400-650): Oil recovers, diversification
    4. Energy crisis (days 650-850): Natural gas spikes
    5. Stabilization (days 850+): Return to balance
    """
    dates = pd.date_range(start=start_date, periods=days, freq='D')

    returns = {'date': dates, 'WTI': [], 'NATGAS': [], 'GOLD': []}

    for i in range(days):
        if i < 250:  # Normal market
            returns['WTI'].append(np.random.normal(0.0005, 0.02))    # 5bp mean, 2% vol
            returns['NATGAS'].append(np.random.normal(0.0003, 0.025))
            returns['GOLD'].append(np.random.normal(0.0002, 0.01))

        elif i < 400:  # COVID crash: oil terrible, gold safe haven
            returns['WTI'].append(np.random.normal(-0.002, 0.04))    # Negative mean, high vol
            returns['NATGAS'].append(np.random.normal(0.0001, 0.03))
            returns['GOLD'].append(np.random.normal(0.001, 0.012))   # Flight to safety

        elif i < 650:  # Recovery: oil stabilizes
            returns['WTI'].append(np.random.normal(0.0008, 0.022))
            returns['NATGAS'].append(np.random.normal(0.0004, 0.02))
            returns['GOLD'].append(np.random.normal(0.0003, 0.01))

        elif i < 850:  # Energy crisis: natural gas spikes
            returns['WTI'].append(np.random.normal(0.0003, 0.025))
            returns['NATGAS'].append(np.random.normal(0.0012, 0.03))  # High returns
            returns['GOLD'].append(np.random.normal(0.0004, 0.012))

        else:  # Stabilization
            returns['WTI'].append(np.random.normal(0.0004, 0.018))
            returns['NATGAS'].append(np.random.normal(0.0005, 0.022))
            returns['GOLD'].append(np.random.normal(0.0003, 0.01))

    df = pd.DataFrame(returns)

    # Compute cumulative returns for visualization
    for commodity in ['WTI', 'NATGAS', 'GOLD']:
        df[f'{commodity}_cumulative'] = (1 + df[commodity]).cumprod() - 1

    return df

# Generate data
df = generate_commodity_returns(days=1000)

# Visualize cumulative returns with regime annotations
plt.figure(figsize=(14, 6))
plt.plot(df['date'], df['WTI_cumulative'], label='WTI Crude', linewidth=2)
plt.plot(df['date'], df['NATGAS_cumulative'], label='Natural Gas', linewidth=2)
plt.plot(df['date'], df['GOLD_cumulative'], label='Gold', linewidth=2)

# Annotate regimes
regime_changes = [df['date'].iloc[250], df['date'].iloc[400],
                  df['date'].iloc[650], df['date'].iloc[850]]
for rc in regime_changes:
    plt.axvline(rc, color='red', linestyle='--', alpha=0.3)

# Regime labels
plt.text(df['date'].iloc[125], 0.15, 'Normal', ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.text(df['date'].iloc[325], 0.15, 'COVID\nCrash', ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))
plt.text(df['date'].iloc[525], 0.15, 'Recovery', ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
plt.text(df['date'].iloc[750], 0.15, 'Energy\nCrisis', ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='orange', alpha=0.5))
plt.text(df['date'].iloc[925], 0.15, 'Stable', ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.title('Commodity Returns with Historical Regime Shifts (Simulated)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Regime structure:")
print("  Days 0-249: Normal market")
print("  Days 250-399: COVID crash (WTI negative, GOLD safe haven)")
print("  Days 400-649: Recovery")
print("  Days 650-849: Energy crisis (NATGAS spikes)")
print("  Days 850-999: Stabilization")

In [ ]:
learning_objectives(["**Commodity markets are inherently non-stationary** — regimes shift due to:", "**Standard bandits fail spectacularly** in non-stationary environments:", "**Adaptive bandits (Discounted TS, Sliding-Window UCB) adapt within 20-50 days:**", "**Hyperparameter tuning is critical:**", "**Real-world insight:** "In commodity trading, non-stationarity isn't the exception — it's the rule.""])

In [ ]:
apply_course_theme()
apply_plot_theme()

## Run Standard Bandit: Watch It Fail

Standard Thompson Sampling will learn the best commodity in the first regime, then get stuck.

In [ ]:
class StandardThompsonSampling:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.alpha = np.ones(n_arms)
        self.beta_params = np.ones(n_arms)

    def select_arm(self):
        samples = [beta.rvs(a, b) for a, b in zip(self.alpha, self.beta_params)]
        return np.argmax(samples)

    def update(self, arm, reward):
        self.alpha[arm] += reward
        self.beta_params[arm] += (1 - reward)

# Map commodities to arms
commodities = ['WTI', 'NATGAS', 'GOLD']

# Normalize returns to [0, 1] for Bernoulli bandit
def normalize_return(ret, scale=0.05):
    """Map return to [0, 1]. Assume returns ~ [-scale, +scale]."""
    return np.clip((ret + scale) / (2 * scale), 0, 1)

# Run standard Thompson Sampling
bandit_std = StandardThompsonSampling(n_arms=3)
selections_std = []
portfolio_returns_std = []

for i in range(len(df)):
    arm = bandit_std.select_arm()
    selections_std.append(arm)

    # Get return for selected commodity
    selected_commodity = commodities[arm]
    true_return = df[selected_commodity].iloc[i]
    portfolio_returns_std.append(true_return)

    # Normalize for bandit update
    reward = normalize_return(true_return)
    bandit_std.update(arm, reward)

# Visualize selections
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Selections
axes[0].scatter(df['date'], selections_std, alpha=0.4, s=3)
for rc in regime_changes:
    axes[0].axvline(rc, color='red', linestyle='--', alpha=0.3)
axes[0].set_ylabel('Selected Commodity')
axes[0].set_yticks([0, 1, 2])
axes[0].set_yticklabels(commodities)
axes[0].set_title('Standard Thompson Sampling: Slow to Adapt')
axes[0].grid(alpha=0.3)

# Cumulative portfolio returns
cumulative_std = (1 + pd.Series(portfolio_returns_std)).cumprod() - 1
axes[1].plot(df['date'], cumulative_std, label='Standard TS Portfolio', linewidth=2)
for rc in regime_changes:
    axes[1].axvline(rc, color='red', linestyle='--', alpha=0.3)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Cumulative Return')
axes[1].set_title('Performance: Gets Stuck on Old Winners')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Standard TS final return: {cumulative_std.iloc[-1]:.2%}")
print("Notice: Algorithm stays with WTI too long during COVID crash!")

## Run Non-Stationary Bandit: Watch It Adapt

Discounted Thompson Sampling with γ=0.95 adapts to regime changes.

In [ ]:
class DiscountedThompsonSampling:
    def __init__(self, n_arms, gamma=0.95):
        self.n_arms = n_arms
        self.gamma = gamma
        self.alpha = np.ones(n_arms)
        self.beta_params = np.ones(n_arms)

    def select_arm(self):
        samples = [beta.rvs(a, b) for a, b in zip(self.alpha, self.beta_params)]
        return np.argmax(samples)

    def update(self, arm, reward):
        self.alpha *= self.gamma
        self.beta_params *= self.gamma
        self.alpha[arm] += reward
        self.beta_params[arm] += (1 - reward)

# Run discounted Thompson Sampling
bandit_disc = DiscountedThompsonSampling(n_arms=3, gamma=0.95)
selections_disc = []
portfolio_returns_disc = []

for i in range(len(df)):
    arm = bandit_disc.select_arm()
    selections_disc.append(arm)

    selected_commodity = commodities[arm]
    true_return = df[selected_commodity].iloc[i]
    portfolio_returns_disc.append(true_return)

    reward = normalize_return(true_return)
    bandit_disc.update(arm, reward)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Selections
axes[0].scatter(df['date'], selections_disc, alpha=0.4, s=3, color='green')
for rc in regime_changes:
    axes[0].axvline(rc, color='red', linestyle='--', alpha=0.3)
axes[0].set_ylabel('Selected Commodity')
axes[0].set_yticks([0, 1, 2])
axes[0].set_yticklabels(commodities)
axes[0].set_title('Discounted Thompson Sampling (γ=0.95): Adapts to Regime Changes')
axes[0].grid(alpha=0.3)

# Cumulative returns
cumulative_disc = (1 + pd.Series(portfolio_returns_disc)).cumprod() - 1
axes[1].plot(df['date'], cumulative_disc, label='Discounted TS Portfolio', linewidth=2, color='green')
axes[1].plot(df['date'], cumulative_std, label='Standard TS (comparison)', linewidth=2, alpha=0.5, linestyle='--')
for rc in regime_changes:
    axes[1].axvline(rc, color='red', linestyle='--', alpha=0.3)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Cumulative Return')
axes[1].set_title('Performance: Adaptive Bandit Outperforms')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Standard TS final return: {cumulative_std.iloc[-1]:.2%}")
print(f"Discounted TS final return: {cumulative_disc.iloc[-1]:.2%}")
print(f"Improvement: {(cumulative_disc.iloc[-1] - cumulative_std.iloc[-1])*100:.2f} percentage points")

## Overlay Regime Indicators on Performance

Visualize how the adaptive bandit shifts allocation as regimes change.

In [ ]:
# Compute allocation percentages over rolling windows
window = 50  # 50-day rolling window

allocation_WTI = []
allocation_NATGAS = []
allocation_GOLD = []

for i in range(len(selections_disc)):
    start = max(0, i - window)
    recent_selections = selections_disc[start:i+1]

    allocation_WTI.append(recent_selections.count(0) / len(recent_selections))
    allocation_NATGAS.append(recent_selections.count(1) / len(recent_selections))
    allocation_GOLD.append(recent_selections.count(2) / len(recent_selections))

# Stacked area plot
plt.figure(figsize=(14, 6))
plt.stackplot(df['date'], allocation_WTI, allocation_NATGAS, allocation_GOLD,
              labels=['WTI', 'NATGAS', 'GOLD'],
              alpha=0.7, colors=['steelblue', 'orange', 'gold'])

# Regime boundaries
for rc in regime_changes:
    plt.axvline(rc, color='red', linestyle='--', alpha=0.5, linewidth=2)

plt.xlabel('Date')
plt.ylabel('Portfolio Allocation (50-day rolling)')
plt.title('Adaptive Allocation: Shifts with Regime Changes')
plt.legend(loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Observations:")
print("- Normal (0-249): Balanced allocation, WTI slightly preferred")
print("- COVID crash (250-399): Rapid shift to GOLD (safe haven)")
print("- Recovery (400-649): Diversification across all three")
print("- Energy crisis (650-849): NATGAS becomes dominant")
print("- Stabilization (850+): Returns to balanced portfolio")

## Modify This: Experiment with Parameters

Try different discount factors and lookback windows to see how they affect adaptation speed.

In [ ]:
from ipywidgets import interact, FloatSlider

def compare_gamma_values(gamma):
    """
    Run discounted TS with specified gamma and visualize performance.
    """
    bandit = DiscountedThompsonSampling(n_arms=3, gamma=gamma)
    portfolio_returns = []

    for i in range(len(df)):
        arm = bandit.select_arm()
        selected_commodity = commodities[arm]
        true_return = df[selected_commodity].iloc[i]
        portfolio_returns.append(true_return)

        reward = normalize_return(true_return)
        bandit.update(arm, reward)

    cumulative = (1 + pd.Series(portfolio_returns)).cumprod() - 1

    # Plot
    plt.figure(figsize=(14, 5))
    plt.plot(df['date'], cumulative, linewidth=2, label=f'γ={gamma:.2f}')
    for rc in regime_changes:
        plt.axvline(rc, color='red', linestyle='--', alpha=0.3)
    plt.xlabel('Date')
    plt.ylabel('Cumulative Return')
    plt.title(f'Portfolio Performance with γ={gamma:.2f} | Final Return: {cumulative.iloc[-1]:.2%}')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Calculate half-life
    half_life = np.log(0.5) / np.log(gamma)
    print(f"Discount factor γ={gamma:.2f}")
    print(f"Half-life: {half_life:.1f} days (data from this many days ago has weight 0.5)")
    print(f"Final cumulative return: {cumulative.iloc[-1]:.2%}")

# Interactive widget
interact(
    compare_gamma_values,
    gamma=FloatSlider(min=0.85, max=0.99, step=0.01, value=0.95, description='γ (gamma):')
);

## Key Insight: Non-Stationarity is the Norm

Let's quantify how often regimes change in commodity markets.

In [ ]:
callout("Non-Stationarity is the Norm", kind="insight")

In [ ]:
# Compute rolling mean and volatility for each commodity
window = 50

for commodity in commodities:
    df[f'{commodity}_rolling_mean'] = df[commodity].rolling(window).mean()
    df[f'{commodity}_rolling_vol'] = df[commodity].rolling(window).std()

# Visualize regime-dependent statistics
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Rolling means
for commodity in commodities:
    axes[0].plot(df['date'], df[f'{commodity}_rolling_mean'], label=commodity, linewidth=2)
for rc in regime_changes:
    axes[0].axvline(rc, color='red', linestyle='--', alpha=0.3)
axes[0].axhline(0, color='black', linestyle='-', linewidth=0.5)
axes[0].set_ylabel('Rolling Mean Return (50-day)')
axes[0].set_title('Mean Returns Shift Across Regimes')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Rolling volatility
for commodity in commodities:
    axes[1].plot(df['date'], df[f'{commodity}_rolling_vol'], label=commodity, linewidth=2)
for rc in regime_changes:
    axes[1].axvline(rc, color='red', linestyle='--', alpha=0.3)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Rolling Volatility (50-day)')
axes[1].set_title('Volatility Surges During Crisis Periods')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("1. Mean returns shift dramatically across regimes (non-stationary)")
print("2. Volatility spikes during crisis periods (heteroskedastic)")
print("3. The 'best' commodity changes: WTI → GOLD → NATGAS → balanced")
print("4. Standard bandits assume stationarity and fail")
print("5. Adaptive bandits are ESSENTIAL for commodity allocation")

## Summary

**What we learned:**

1. **Commodity markets are inherently non-stationary** — regimes shift due to:
   - Economic shocks (COVID, recessions)
   - Geopolitical events (energy crises, wars)
   - Seasonal patterns
   - Supply/demand imbalances

2. **Standard bandits fail spectacularly** in non-stationary environments:
   - Get stuck on historically good assets
   - Miss regime shifts for 100+ days
   - Accumulate large regret

3. **Adaptive bandits (Discounted TS, Sliding-Window UCB) adapt within 20-50 days:**
   - Weight recent data more heavily
   - Continuously re-explore
   - Recover from regime shifts

4. **Hyperparameter tuning is critical:**
   - Lower γ (0.90) → fast adaptation, higher variance
   - Higher γ (0.97) → slow adaptation, more stable
   - Calibrate to expected regime duration

5. **Real-world insight:** "In commodity trading, non-stationarity isn't the exception — it's the rule."

**Practical takeaways:**
- Always use non-stationary algorithms for commodity allocation
- Backtest on historical data with known regime changes
- Combine with change-point detection for faster adaptation
- Monitor rolling statistics to detect regime shifts

**Next steps:**
- Module 6 Exercises: Implement these algorithms yourself
- Combine with contextual features (Module 3) for even better performance
- Production deployment (Module 7): Real-time regime monitoring

In [ ]:
key_takeaways(["Commodity markets are inherently non-stationary", "Standard bandits fail spectacularly", "Adaptive bandits (Discounted TS, Sliding-Window UCB) adapt within 20-50 days:", "Hyperparameter tuning is critical:", "Real-world insight:"])